# 模块三：报名与取消 —— 交互式演示与验证

本 Notebook 用 VS Code 的 Jupyter 插件逐格演示第三部分（报名与取消）的业务规则，并可直接跑通该模块的测试与覆盖率。

**运行前提**

1. 用 VS Code 打开仓库根目录（含 `pyproject.toml` 的那一层）。
2. 右下角 `Select Kernel` → `Python Environments` → 选择项目的 `.venv`。
3. 从上到下依次执行单元格（`Shift+Enter`）。

**约定**：本 Notebook 只调用模块三的公开类（`RegistrationRepository`、`RegistrationService`），
不修改任何公共代码；学生与活动读取用两个小替身模拟，正是生产环境里由模块一、模块二提供的协议实现。


## 0. 环境引导

先定位仓库根目录并加入 `sys.path`，这样无论在仓库哪一层打开 Notebook 都能 `import campusApp`。


In [ ]:
import pathlib
import sys

cwd = pathlib.Path.cwd()
projectRoot = next(
    (candidate for candidate in [cwd, *cwd.parents]
     if (candidate / "pyproject.toml").exists() and (candidate / "campusApp").is_dir()),
    None,
)
if projectRoot is None:
    raise RuntimeError("未找到仓库根目录，请在 stusent-event 仓库内运行本 Notebook")
if str(projectRoot) not in sys.path:
    sys.path.insert(0, str(projectRoot))

print("仓库根目录:", projectRoot)
print("Python:", sys.version.split()[0])


## 1. 模块依赖一览

```
RegistrationView   ← 只做界面，不碰共享存储
      ↓
RegistrationService ← 校验顺序、状态判定、递补
      ↓                ↘ StudentReader（模块一）  ActivityReader（模块二）
RegistrationRepository → 只读写 state.registrations 与 nextRegistrationId
      ↓
MemoryStore（core，共享唯一实例；runWrite 原子发布）
```

构造签名固定为 `RegistrationService(store, repository, studentReader, activityReader, clock)`。


In [ ]:
from campusApp.core.MemoryStore import MemoryStore
from campusApp.core.constants import (
    MAX_ACTIVE_REGISTRATIONS,
    STATUS_CANCELLED,
    STATUS_CONFIRMED,
    STATUS_WAITLISTED,
)
from campusApp.core.contracts import Activity, Student
from campusApp.core.timeUtils import formatDateTime, parseDateTime
from campusApp.registrations.RegistrationRepository import RegistrationRepository
from campusApp.registrations.RegistrationService import RegistrationService

print("个人有效报名上限:", MAX_ACTIVE_REGISTRATIONS)
print("状态码:", STATUS_CONFIRMED, "/", STATUS_WAITLISTED, "/", STATUS_CANCELLED)


## 2. 替身与打印工具

- `StudentLookup` / `ActivityLookup`：实现 `core.contracts` 里的两个只读协议，生产环境分别由学生模块与活动模块提供。
- `FakeClock`：可调时钟，用来演示时间边界。
- `seed` / `showRegistrations`：构造前置内存状态、打印报名表。


In [ ]:
class StudentLookup:
    """学生读取替身；生产环境由模块一的 StudentRepository 实现同一协议。"""

    def findById(self, state, studentId):
        return state.students.get(studentId)


class ActivityLookup:
    """活动读取替身；生产环境复用活动模块的 ActivityRepository。"""

    def findById(self, state, activityId):
        return state.activities.get(activityId)


class FakeClock:
    """可调时钟，测试与演示都用它固定 now。"""

    def __init__(self, now):
        self.current = now

    def now(self):
        return self.current


def seed(store, students, activities):
    """把学生与活动写入共享存储，并把编号计数器推到最大值之后。"""
    def write(state):
        for student in students:
            state.students[student.studentId] = student
        for activity in activities:
            state.activities[activity.activityId] = activity
        state.nextStudentId = max(state.students, default=0) + 1
        state.nextActivityId = max(state.activities, default=0) + 1
        state.nextRegistrationId = max(state.registrations, default=0) + 1
    store.runWrite(write)


def showRegistrations(store):
    """打印当前内存中的全部报名记录。"""
    def read(state):
        rows = []
        for item in sorted(state.registrations.values(), key=lambda record: record.registrationId):
            activity = state.activities.get(item.activityId)
            rows.append((
                item.registrationId,
                item.studentId,
                activity.title if activity else "-",
                item.status,
                formatDateTime(item.joinedAt),
            ))
        return rows

    rows = store.runRead(read)
    if not rows:
        print("（暂无报名记录）")
        return
    header = f"{'编号':<6}{'学生':<6}{'活动':<14}{'状态':<12}{'报名时间'}"
    print(header)
    print("-" * len(header))
    for row in rows:
        print(f"{row[0]:<6}{row[1]:<6}{row[2]:<14}{row[3]:<12}{row[4]}")


print("工具已就绪")


## 3. 准备演示数据

时间基准固定为 2027-03-01 10:00（北京时间），活动容量为 **1**，方便马上演示"名额已满 → 候补"。


In [ ]:
NOW = parseDateTime("2027-03-01 10:00:00", "now")
OPEN_AT = NOW - 3600          # 一小时前开始报名
CLOSE_AT = NOW + 3600         # 一小时后截止
START_AT = NOW + 7200         # 两小时后活动开始
WINDOW_SECONDS = CLOSE_AT - OPEN_AT


def makeActivity(activityId, title, capacity, isClosed=False):
    """按统一时间窗口构造活动，结束时间由时长推导。"""
    return Activity(
        activityId, title, capacity, 60,
        OPEN_AT, CLOSE_AT, START_AT, START_AT + 3600,
        isClosed, NOW,
    )


store = MemoryStore()
clock = FakeClock(NOW)
service = RegistrationService(
    store, RegistrationRepository(), StudentLookup(), ActivityLookup(), clock
)

seed(
    store,
    [Student(index, f"2023000{index:03d}", f"学生{index}", 2, NOW) for index in (1, 2, 3)],
    [makeActivity(1, "校园音乐会", 1)],
)

print("now          :", formatDateTime(NOW))
print("报名窗口     :", formatDateTime(OPEN_AT), "→", formatDateTime(CLOSE_AT))
print("活动开始     :", formatDateTime(START_AT))
print("活动容量     : 1")


## 4. 报名：正式还是候补

判定条件只有一条：`已确认人数 < 容量` 时正式报名，否则进入候补。剩余名额随结果一起返回。


In [ ]:
for studentId in (1, 2):
    result = service.registerStudent(studentId, 1)
    if result.ok:
        label = service.describeStatus(result.data.registration)
        print(f"学生{studentId} → {label}，报名编号 {result.data.registration.registrationId}，"
              f"剩余名额 {result.data.remainingSeats}")
    else:
        print(f"学生{studentId} → 失败：{result.error.message}")

print()
showRegistrations(store)


## 5. 取消与自动递补

取消**正式**名额时，会在同一次写事务里把该活动 `registrationId` 最小的候补转成正式，并返回被递补的编号。
取消**候补**不会触发递补。


In [ ]:
result = service.cancelRegistration(1, 1)
if result.ok:
    promoted = result.data.promotedRegistrationId
    print(f"学生1 取消报名编号 1 → 成功；被递补的候补编号：{promoted}")
else:
    print("取消失败：", result.error.message)

print()
showRegistrations(store)
print()
print("注意：学生2 从 WAITLISTED 变成了 CONFIRMED，这是递补的结果。")


## 6. 重复报名与个人额度

- 同一学生对同一活动只能有一条未取消记录（候补也算），取消后才能重新申请。
- 有效报名上限 **3 条**，跨活动累计；活动结束后按 `endAt > now` 自动让出额度。


In [ ]:
again = service.registerStudent(1, 1)
print("学生1 取消后重新申请:",
      "成功 → " + service.describeStatus(again.data.registration) if again.ok else again.error.message)

duplicate = service.registerStudent(1, 1)
print("学生1 立刻再报同一活动:",
      "成功" if duplicate.ok else f"{duplicate.error.code} / {duplicate.error.message}")

# 再补两个活动，用来撞"3 条额度"的上限
def addActivities(activityIds):
    def write(state):
        for activityId in activityIds:
            state.activities[activityId] = makeActivity(activityId, f"活动{activityId}", 5)
            state.nextActivityId = activityId + 1
    store.runWrite(write)


addActivities([2, 3, 4])
for activityId in (2, 3):
    outcome = service.registerStudent(2, activityId)
    print(f"学生2 报名活动{activityId}:", "成功" if outcome.ok else outcome.error.message)

over = service.registerStudent(2, 4)
print("学生2 的第 4 条:", "成功" if over.ok else f"{over.error.code} / {over.error.message}")


## 7. 时间边界：报名窗口是左闭右开

`openAt <= now < closeAt`。下面每个用例都用一份**全新的** Store，避免相互干扰。


In [ ]:
def boundaryCheck(relativeSeconds):
    """以 openAt 为基准偏移若干秒，返回该时刻能否报名。"""
    freshStore = MemoryStore()
    freshService = RegistrationService(
        freshStore, RegistrationRepository(), StudentLookup(), ActivityLookup(),
        FakeClock(OPEN_AT + relativeSeconds),
    )
    seed(freshStore, [Student(1, "2023000001", "学生1", 2, NOW)], [makeActivity(1, "边界活动", 5)])
    outcome = freshService.registerStudent(1, 1)
    return "允许" if outcome.ok else f"拒绝（{outcome.error.code}）"


cases = [
    (-1, "开始前 1 秒"),
    (0, "恰好开始"),
    (WINDOW_SECONDS - 1, "截止前 1 秒"),
    (WINDOW_SECONDS, "恰好截止"),
    (WINDOW_SECONDS + 1, "截止后 1 秒"),
]
print(f"{'时刻':<14}{'结果'}")
print("-" * 34)
for offset, label in cases:
    print(f"{label:<14}{boundaryCheck(offset)}")


## 8. 并发：两个线程抢最后一个名额

`MemoryStore.runWrite` 用同一把锁串行执行写回调，后进入的线程读到的是前一个已发布的状态，因此不会超额。


In [ ]:
import threading

raceStore = MemoryStore()
raceService = RegistrationService(
    raceStore, RegistrationRepository(), StudentLookup(), ActivityLookup(), FakeClock(NOW)
)
seed(raceStore, [Student(index, f"2023000{index:03d}", f"学生{index}", 2, NOW) for index in (1, 2)],
     [makeActivity(1, "最后名额", 1)])

barrier = threading.Barrier(2)
outcomes = []
lock = threading.Lock()


def apply(studentId):
    barrier.wait()                      # 两个线程尽量同时进入
    outcome = raceService.registerStudent(studentId, 1)
    with lock:
        outcomes.append((studentId, outcome.data.registration.status if outcome.ok else outcome.error.code))


threads = [threading.Thread(target=apply, args=(studentId,)) for studentId in (1, 2)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

for studentId, status in sorted(outcomes):
    print(f"学生{studentId} → {status}")
print()
showRegistrations(raceStore)
print()
print("结论：容量 1 时恰好一条 CONFIRMED、一条 WAITLISTED。")


## 9. 跑通测试

直接调用当前内核的解释器执行 `pytest tests/registrations`。输出会打印在本单元下方。


In [ ]:
import importlib.util
import subprocess

if importlib.util.find_spec("pytest") is None:
    print("当前内核缺少 pytest，请先执行：python -m pip install -r requirements-dev.txt")
else:
    exitCode = subprocess.call(
        [sys.executable, "-m", "pytest", "tests/registrations",
         "-q", "--no-header", "-p", "no:cacheprovider"],
        cwd=str(projectRoot),
    )
    print("\npytest 退出码:", exitCode, "（0 表示全部通过）")


## 10. 覆盖率

要求核心业务逻辑覆盖率 ≥ 80%，实际为：Repository 100%、Service 98%、View 86%。


In [ ]:
import importlib.util
import subprocess

if importlib.util.find_spec("pytest_cov") is None:
    print("当前内核缺少 pytest-cov，请先执行：python -m pip install -r requirements-dev.txt")
else:
    exitCode = subprocess.call(
        [sys.executable, "-m", "pytest", "tests/registrations",
         "--cov=campusApp.registrations", "--cov-report=term-missing",
         "--cov-fail-under=80", "-q", "--no-header", "-p", "no:cacheprovider"],
        cwd=str(projectRoot),
    )
    print("\n覆盖率统计退出码:", exitCode)


## 11. 自查清单（对照协作说明第 7 节）

| 检查项 | 状态 |
| --- | --- |
| 只依赖 `core.contracts` + 两个 Reader 协议，不 import 他人 Service/View | ✅ |
| 校验顺序：参数 → 学生存在 → 活动存在 → 报名窗口/关闭 → 重复 → 额度 → 容量 | ✅ `registerStudent` |
| 取消顺序：参数 → 记录存在 → 所属学生 → 是否已取消 → 活动是否已开始 | ✅ `cancelRegistration` |
| 个人额度按"活动 `endAt > now`"统计，已取消不计 | ✅ 用例表中两行专门覆盖 |
| 候补按编号 FIFO，取消正式立即递补一条 | ✅ |
| 递补不再检查额度 | ✅ 有专门用例 |
| 失败丢弃整份草稿、不消耗编号 | ✅ 注入异常的用例 |
| 覆盖率 ≥ 80% | ✅ 92% |
| 提交范围只含三个白名单目录（代码、测试、文档） | ✅ 提交前用 `git diff --cached --name-only` 复核 |

**下一步**：等模块一提供 `StudentRepository`、模块四提供 `QueryService` 后，
由集成人在 `app.py` 挂载 `RegistrationView`。缺少项见同目录的 `change-requests.md`。
